#### 1.Download data

In [3]:
import pandas as pd
import numpy as np
import optuna
from category_encoders import CountEncoder
from collections import Counter
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.metrics import mean_squared_error
from sklearn.metrics import roc_auc_score

/Users/vadimbatalev/Documents/programming/s21/base/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
data = pd.read_csv('../datasets/training.csv')
data.PurchDate = pd.to_datetime(data['PurchDate'])
data.sort_values(by='PurchDate',ascending=True,inplace=True)

In [15]:
num = (len(data)//3)
X_train = data[0:num]
print(X_train.shape)
X_val = data[num:2*num]
print(X_val.shape)
X_test = data[2*num:len(data)]
print(X_test.shape)

(24327, 34)
(24327, 34)
(24329, 34)


In [16]:
X_train = data[data['PurchDate'] <= '2009-09-15']
X_val = data[(data['PurchDate'] > '2009-09-15') & (data['PurchDate'] <= '2010-05-14')]
X_test = data[data['PurchDate'] > '2010-05-14']

In [17]:
def extract_date_features(df):
    df = df.copy()
    df['day'] = df['PurchDate'].dt.day
    df['month'] = df['PurchDate'].dt.month
    df = df.drop('PurchDate', axis=1)
    return df

X_train_object = extract_date_features(X_train)
X_val_object = extract_date_features(X_val)
X_test_object = extract_date_features(X_test)

In [ ]:
object_columns = []
for column in X_train.columns:
    if ((X_train[column].dtypes))=='object':
        object_columns.append(column)

In [ ]:
encoder = CountEncoder(cols=object_columns,handle_missing='value')
encoder.fit(X_train_object)

X_train_enc = encoder.transform(X_train_object)
X_val_enc = encoder.transform(X_val_object)
X_test_enc = encoder.transform(X_test_object)

,verbose,0
,cols,"['Auction', 'Make', ...]"
,drop_invariant,False
,return_df,True
,handle_unknown,'value'
,handle_missing,'value'
,min_group_size,None
,combine_min_nan_groups,True
,min_group_name,None
,normalize,False


In [ ]:
train_mean = X_train_enc.mean()
X_train_enc = X_train_enc.fillna(train_mean)
X_val_enc = X_val_enc.fillna(train_mean)
X_test_enc = X_test_enc.fillna(train_mean)

y_train = X_train_enc['IsBadBuy'].reset_index(drop=True)
X_train_enc.drop('IsBadBuy',axis=1,inplace=True)
y_val = X_val_enc['IsBadBuy'].reset_index(drop=True)
X_val_enc.drop('IsBadBuy',axis=1,inplace=True)
y_test = X_test_enc['IsBadBuy'].reset_index(drop=True)
X_test_enc.drop('IsBadBuy',axis=1,inplace=True)

#### 2. Create a Python class for Decision Tree Classifier and Decision Tree Regressor (MSE loss).

In [24]:
def gini(y_true,y_predict):
    roc_auc = roc_auc_score(y_score=y_predict,y_true=y_true)
    return (2 * roc_auc - 1)

In [369]:
class Node:
    def __init__(self, feature_idx=None, treshold=None, info_gain=None, left=None, right=None, value=None):
        # Decision Node
        self.feature_idx = feature_idx
        self.treshold = treshold
        self.info_gain = info_gain
        self.left = left
        self.right = right
        #leaf
        self.value = value

In [ ]:
class DecisionTree:
    def __init__(self, max_depth = 2):
        self.max_depth = max_depth

    def build_tree(self,dataset,curr_depth=0):
        X, y = dataset[:, :-1], dataset[:, -1]
        n_samples, n_features = X.shape

        if curr_depth <= self.max_depth:
            best_split = self.best_split(dataset,n_features)

            if best_split['info_gain'] > 0:
                left_node = self.build_tree(best_split['left_dataset'],curr_depth + 1)
                right_node = self.build_tree(best_split['right_dataset'],curr_depth + 1)

                return Node(best_split['feature_idx'], best_split['treshold'],best_split['info_gain'], left_node, right_node)
            
        class_counts = Counter(y)
        leaf_value = class_counts.get(1,0) / len(y)
        return Node(value=leaf_value)
    
    def best_split(self, dataset, n_features):
        best_split = {'feature_idx' : None,'treshold' : None, 'info_gain' : -1, 'left_dataset' : None, 'right_dataset' : None}

        for feature_idx in range(n_features):
            feature_values = dataset[:, feature_idx]
            tresholds = np.unique(feature_values)

            for treshold in tresholds:
                left_dataset, right_dataset = self.split(dataset, feature_idx, treshold)

                if len(left_dataset) and len(right_dataset):
                    parent_y, left_y, right_y = dataset[:, -1], left_dataset[:,-1], right_dataset[:,-1]

                    info_gain = self.information_gain(parent_y, left_y, right_y)

                    if info_gain > best_split['info_gain']:
                        best_split['feature_idx']= feature_idx
                        best_split['treshold'] = treshold
                        best_split['info_gain'] = info_gain
                        best_split['left_dataset'] = left_dataset
                        best_split['right_dataset'] = right_dataset

        return best_split
    
    def split(self, dataset, feature_idx, treshold):
        left_mask = dataset[:, feature_idx] <= treshold
        left_dataset = dataset[left_mask]
        right_dataset = dataset[~left_mask]
        return left_dataset, right_dataset

    
    def information_gain(self, parent_y, left_y, right_y):
        left_weight = len(left_y) / len(parent_y)
        right_weight = len(right_y) / len(parent_y)

        information_weight = self.gini(parent_y) - (left_weight * self.gini(left_y) + right_weight * self.gini(right_y))

        return information_weight
    
    def gini(self, y):
        gini = 0

        class_labels = np.unique(y)
        for class_label in class_labels:
            p = len(y[y == class_label]) / len(y)
            gini += p * (1-p)

        return gini
    
    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        dataset = np.concatenate([X,y.reshape(-1,1)],axis=1)
        self.root = self.build_tree(dataset)

    def predict(self,X):
        probabilities = self.predict_proba(X)
        return (probabilities >= 0.5).astype(int)
    
    def predict_proba(self,X):
        probabilities = [self.predict_class(row, self.root) for row in X]
        return np.array(probabilities)
    
    def predict_class(self, row, node):
        if node.value != None:
            return node.value
        
        feature_val = row[node.feature_idx]
        if feature_val <= node.treshold:
            return self.predict_class(row, node.left)
        else:
            return self.predict_class(row, node.right)

In [ ]:
class MyExtraRandomizedDecisionTree:
    def __init__(self, max_depth = 2, n_random_split = 10, random_state = 21):
        self.max_depth = max_depth
        self.n_random_split = n_random_split
        self.random_state = random_state
        self.random_values = np.random.RandomState(random_state)

    def build_tree(self,dataset,curr_depth=0):
        X, y = dataset[:, :-1], dataset[:, -1]
        n_samples, n_features = X.shape

        if curr_depth <= self.max_depth:
            best_split = self.best_split(dataset,n_features)

            if best_split['info_gain'] > 0:
                left_node = self.build_tree(best_split['left_dataset'],curr_depth + 1)
                right_node = self.build_tree(best_split['right_dataset'],curr_depth + 1)

                return Node(best_split['feature_idx'], best_split['treshold'],best_split['info_gain'], left_node, right_node)
            
        class_counts = Counter(y)
        leaf_value = class_counts.get(1,0) / len(y)
        return Node(value=leaf_value)
    
    def best_split(self, dataset, n_features):
        best_split = {'feature_idx' : None,'treshold' : None, 'info_gain' : -1, 'left_dataset' : None, 'right_dataset' : None}

        for feature_idx in range(n_features):
            feature_values = dataset[:, feature_idx]
            min_val = feature_values.min()
            max_val = feature_values.max()
            if min_val == max_val:
                continue
            random_tresholds = self.random_values.uniform(min_val, max_val, self.n_random_split)

            for treshold in random_tresholds:
                left_dataset, right_dataset = self.split(dataset, feature_idx, treshold)

                if len(left_dataset) and len(right_dataset):
                    parent_y, left_y, right_y = dataset[:, -1], left_dataset[:,-1], right_dataset[:,-1]

                    info_gain = self.information_gain(parent_y, left_y, right_y)

                    if info_gain > best_split['info_gain']:
                        best_split['feature_idx']= feature_idx
                        best_split['treshold'] = treshold
                        best_split['info_gain'] = info_gain
                        best_split['left_dataset'] = left_dataset
                        best_split['right_dataset'] = right_dataset

        return best_split
    
    def split(self, dataset, feature_idx, treshold):
        left_mask = dataset[:, feature_idx] <= treshold
        left_dataset = dataset[left_mask]
        right_dataset = dataset[~left_mask]
        return left_dataset, right_dataset

    
    def information_gain(self, parent_y, left_y, right_y):
        left_weight = len(left_y) / len(parent_y)
        right_weight = len(right_y) / len(parent_y)

        information_weight = self.gini(parent_y) - (left_weight * self.gini(left_y) + right_weight * self.gini(right_y))

        return information_weight
    
    def gini(self, y):
        gini = 0

        class_labels = np.unique(y)
        for class_label in class_labels:
            p = len(y[y == class_label]) / len(y)
            gini += p * (1-p)

        return gini
    
    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        dataset = np.concatenate([X,y.reshape(-1,1)],axis=1)
        self.root = self.build_tree(dataset)

    def predict(self,X):
        probabilities = self.predict_proba(X)
        return (probabilities >= 0.5).astype(int)
    
    def predict_proba(self,X):
        X = np.array(X)
        probabilities = [self.predict_class(row, self.root) for row in X]
        return np.array(probabilities)
    
    def predict_class(self, row, node):
        if node.value != None:
            return node.value
        
        feature_val = row[node.feature_idx]
        if feature_val <= node.treshold:
            return self.predict_class(row, node.left)
        else:
            return self.predict_class(row, node.right)

In [ ]:
model = DecisionTree(max_depth=7)
model.fit(X_train_enc,y_train)

y_predict = model.predict_proba(np.array(X_val_enc))
gini(y_val,y_predict)

0.40240237380727684

In [408]:
model = MyExtraRandomizedDecisionTree(n_random_split=10,max_depth=5)
model.fit(X_train_enc,y_train)

In [409]:
y_predict = model.predict_proba(X_val_enc)
gini(y_val,y_predict)

0.40128259556449253

#### 4. Use sklearn's DecisionTreeClassifier

In [418]:
sk_model = DecisionTreeClassifier(max_depth=7)
sk_model.fit(X_train_enc,y_train)
y_predict = sk_model.predict_proba(X_val_enc)[:,1]
gini(y_val,y_predict)

0.4015338471281753

#### 5. Implement the RandomForestClassifie

In [436]:
model = RandomForestClassifier(max_depth=6,random_state=21)

model.fit(X_train_enc,y_train)
y_predict = model.predict_proba(X_val_enc)[:,1]
gini(y_predict=y_predict,y_true=y_val)

0.46818639220816927

In [454]:
class MyRandomForest:
    def __init__(self,n_trees = 10, max_depth = 5,random_state = 21):
        self.n_trees = n_trees
        self.max_depth = max_depth
        #self.n_features = n_features
        self.random_state = random_state
        self.randomseed = np.random.RandomState(random_state)
        self.trees = []

    def fit(self,X,y):
        X = np.array(X)
        y = np.array(y)
        self.trees = []
        for _ in range(self.n_trees):
            tree = DecisionTree(max_depth=self.max_depth)
            X_sample, y_sample = self.bootstrap_samples(X,y)
            tree.fit(X_sample,y_sample)
            self.trees.append(tree) 

    def bootstrap_samples(self,X, y):
        n_samples = X.shape[0]
        idx = self.randomseed.choice(n_samples,n_samples, replace=True)
        return X[idx],y[idx]
    
    def most_common(self, y):
        counter = Counter(y)
        most_common = counter.most_common(1)[0][0]
        return most_common   
    
    def predict(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(predictions, 0, 1)
        return np.array([self.most_common(pred) for pred in tree_preds])
    
    def predict_proba(self, X):
        all_proba = np.array([tree.predict_proba(X) for tree in self.trees])
        mean_proba = np.mean(all_proba, axis=0)
        
        return mean_proba

In [ ]:
model = MyRandomForest()
X_train = X_train_enc.iloc[:10000]
y_train = y_train.iloc[:10000]
model.fit(X_train,y_train)

predict = model.predict_proba(np.array(X_val_enc.iloc[:10000]))
gini(y_predict=predict,y_true=y_val.iloc[:10000])

0.4377169445578526

In [ ]:
class GBDTClassifier:
    def __init__(self, max_depth=5, number_of_trees=10, max_features=None, learning_rate=0.5):
        self.max_depth = max_depth
        self.number_of_trees = number_of_trees
        self.max_features = max_features
        self.learning_rate = learning_rate
        self.trees = []
        self.initial_prediction = None
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
    
    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        self.trees = []

        positive_ratio = np.mean(y)
        self.initial_prediction = np.log(positive_ratio / (1 - positive_ratio + 1e-8))
        current_predictions = np.full(len(y), self.initial_prediction)

        for i in range(self.number_of_trees):
            probabilities = self.sigmoid(current_predictions)
            residuals = y - probabilities
            
            tree = DecisionTree(max_depth=self.max_depth)
            tree.fit(X, residuals)
            tree_predictions = tree.predict(X)
            current_predictions += self.learning_rate * tree_predictions
            
            self.trees.append(tree)
    
    def predict_proba(self, X):
        X = np.array(X)
        
        predictions = np.full(X.shape[0], self.initial_prediction)
        
        for tree in self.trees:
            predictions += self.learning_rate * tree.predict(X)
        
        probabilities = self.sigmoid(predictions)
        return probabilities
    
    def predict(self, X):
        probabilities = self.predict_proba(X)
        return (probabilities >= 0.5).astype(int)


In [575]:
model = GBDTClassifier()
model.fit(np.array(X_train_enc[:1000]),np.array(y_train[:1000]))
predict = model.predict_proba(X_val_enc[:1000])
gini(y_predict=np.array(predict),y_true=np.array(y_val[:1000]))

0.3460123069498069

#### 7. Use LightGBM, Catboost, and XGBoost 

##### lightgbm - microsoft, от слова light.  
1 подход - goss. Выбор состоит не из всех элементов, а часть. \
Модель хочет научиться на элементах, которые не похоже друг на друга, но при этом не забыть и про элементы, которые похоже друг на друга. \
Это типо нормисы и ненормисы из irl.  
Мы получаем какое-то количество похожих элементов и непохожих элементов. Из 2-ух таких мини датасетов мы выбираем топ 5% объектов по перцентрилю.  \
Таким образом, мы обучаемся не на всех объектов, а только на некоторых, поэтому деревья будут строиться быстрее. 


2 подход - EFB. Его ключевая мысль - это работа с взаимоисключащюими признаками(пример с полом). \
Образно говоря, lightGBM объединяет все такие признаки в один банд, то есть как будто в 1 фичу.
В работе lightgbm деревья строятся несбалансированно. \
Он использует подход leaf_wise, когда на каждом шаге выбирается лист с максимальным приростом функции качества и делится именно он. \
__когда он нам нужен?__ когда нужно быстро построить модель градиентного бустинга и посмотреть применим ли вообще бустинг. Также может быть хорош для оптимизации гиперпараметров

##### catboost - любимый яндекс

Катбуст выделяется тем, что может без какой-либо помощи кодировщиков обрабатывать категориальные данные.  \
При этом, мы можем выбрать столбцы с категориальными признаками сами, либо за нас это сделает сам катбуст.  
Причем делает это очень умно, она кодирует разные признаки по разному, тут используется такое понятие, как "счетчик".  
Разные форматы категориальных данных кодируются по разному.  
Еще можно выделить наличие регуляризации. Катбуст использует L2 и можно в качестве параметра выбрать силу этого регуляризатор.  
Касательно работы с категориальными данными, катбуст использует Ordered Target Statistics, позволяющую предотвратить утечку данных. 
Для кодировки категориальных признаков, модель использует упорядоченное целевое кодирование - Ordered Target Encoding. 
Для каждого i-го объекта с категорией c, CatBoost вычисляет:
$$  encoding_i = \frac{(count_{pos} + prior*alfa)}{count_{pos} + alfa}$$
__count_positive__ — количество объектов с категорией $c$ и положительным таргетом среди строк с индексами < i.  
__count_total__ — общее количество объектов с категорией $c$ среди строк с индексами < i.  
__prior__ — общее среднее значение таргета по всему датасету (априорное значение). \
__alfa__ — параметр сглаживания (обычно 1-10)

##### XGBoost - нестареющая классика

XGBoost обладает очень сильным контролем над переобучением. Имеет L1,L2 регуляризации.  
Касаемо построение деревьев, на каждом уровне он расширяет деревья симметрично и сбалансированно(во все стороны).  
XGBoost очень хорошо оптимизирован на системном уровне. Он использует параллелизацию построения деревьев, кэш-оптимизацию.  
Эффективно работает с sparse матрицами и хорошо обрабатывает пропуски.  
Вообще, XGBoost напоминает песочницу. Ты сам собираешь свою модель с регуляризаторами, с тонкой ручной настройку гиперпараметров.  
Стоит также упомянуть про DART режим.  
Dart режим - это специальный режим работы XGBoost, когда на каждой итерации обучения случайно выбираются и временно отключаются некоторые деревья.  
Это чем-то напоминает dropout у нейронных сетей.  
Этот режим используют в случаях, когда модель склонна к переобучению. Иногда может дать лучше метрику, чем обычный режим. 

In [67]:
def optimize_model(model_name, X_train, y_train, X_val, y_val, n_trials=50,random_state=21):
    
    def objective(trial):
        params = {
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        }

        if model_name == 'lightgbm':
            params['num_leaves'] = trial.suggest_int('num_leaves', 20, 100)
            params['min_child_samples'] = trial.suggest_int('min_child_samples', 5, 100)
            params['subsample'] = trial.suggest_float('subsample', 0.5, 1.0)
            params['reg_alpha'] = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True)
            params['reg_lambda'] = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)
            params['min_split_gain'] = trial.suggest_float('min_split_gain', 0.0, 1.0)
            model = LGBMClassifier(**params, verbose=-1,random_state=random_state)
            
        elif model_name == 'xgboost':
            params['min_child_weight'] = trial.suggest_int('min_child_weight', 1, 10)
            params['subsample'] = trial.suggest_float('subsample', 0.5, 1.0)
            params['reg_alpha'] = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True)
            params['reg_lambda'] = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)
            model = XGBClassifier(**params, verbosity=0,random_state=random_state)
            
        elif model_name == 'catboost':
            params['iterations'] = params.pop('n_estimators')
            params['l2_leaf_reg'] = trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True)
            params['random_strength'] = trial.suggest_float('random_strength', 0.0, 10.0)
            params['border_count'] = trial.suggest_int('border_count', 32, 255)
            params['min_data_in_leaf'] = trial.suggest_int('min_data_in_leaf', 1, 100)
            params['leaf_estimation_iterations'] = trial.suggest_int('leaf_estimation_iterations', 1, 10)
            model = CatBoostClassifier(**params, verbose=False,random_state=random_state)

        elif model_name== 'catboost_object':
            params['iterations'] = params.pop('n_estimators')
            params['l2_leaf_reg'] = trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True)
            model = CatBoostClassifier(**params, verbose=False,cat_features=object_columns,random_state=random_state)
     
        model.fit(X_train, y_train)
        y_pred = model.predict_proba(X_val)[:, 1]
        score = gini(y_val, y_pred)
        
        return score
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    print(f"\n{model_name.upper()}:")
    print(f"Лучший Gini: {study.best_value:.4f}")
    print(f"Лучшие параметры: {study.best_params}\n")
    
    return study.best_params, study.best_value


In [ ]:
for model_name in ['lightgbm', 'xgboost', 'catboost']:
    best_params,best_metric = optimize_model(model_name, X_train_enc, y_train, X_val_enc, y_val, n_trials=50)

#### 8. Take the best mode

In [69]:
model = XGBClassifier(**best_params,random_state=21)
model.fit(X_train_enc,y_train)

/Users/vadimbatalev/Documents/programming/s21/base/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [20:42:59] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "border_count", "l2_leaf_reg", "leaf_estimation_iterations", "min_data_in_leaf", "random_strength" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [70]:
predict = model.predict_proba(X_train_enc)[:,1]
metric_train = gini(y_predict=predict,y_true=y_train)
predict = model.predict_proba(X_val_enc)[:,1]
metric_val = gini(y_predict=np.array(predict),y_true=y_val)
predict = model.predict_proba(X_test_enc)[:,1]
metric_test = gini(y_predict=predict,y_true=y_test)

In [71]:
print(f'gini train - {metric_train}\n gini val - {metric_val}\n gini test - {metric_test}')

gini train - 0.6171284982994092
 gini val - 0.4721519622781345
 gini test - 0.4893916098696973
